# Stanford CS231N | Spring 2025 | Lecture 11: Large Scale Distributed Training

<p align="center"><img src="./lecture_11_slides/slide_4_00-00-00.133.jpg" width="75%" alt="Lecture Video at 00:00:00.133" /></p>

Welcome back to CS231n lecture 11.


<p align="center"><img src="./lecture_11_slides/slide_320_00-00-10.677.jpg" width="75%" alt="Lecture Video at 00:00:10.677" /></p>

Today, we're going to talk about large-scale distributed training. This is a pretty exciting topic, because this is basically how all neural networks get trained in practice today. When you look at large models from startups, from industries, even in academia, really large-scale is the new norm in deep learning nowadays. That's actually something that's changed quite a lot in the last 10 years since we started this class.

Nowadays, the new norm is to train models on tens, hundreds, thousands, even tens of thousands of devices concurrently. So we need to develop new algorithms and new ways of thinking in order to do that.


<p align="center"><img src="./lecture_11_slides/slide_1896_00-01-03.263.jpg" width="75%" alt="Lecture Video at 00:01:03.263" /></p>

There are a lot of really amazing, powerful models that have been trained in the last couple of years, from Google, from OpenAI, from Anthropic, from others. But basically they don't share any details whatsoever about their models anymore. There's a very famous quote that marked a sea change in the industry to me; that was in the GPT-4 paper back in 2023.

That's basically the news. That's basically been the state-of-the-art for large-scale models these last three years since GPT-4. They don't tell you anything about anything. They'll tell you nothing about the model; you'll be lucky if they'll tell you it's a transformer.

Llama3 is notable, not because it's the best model out there, but because it's one of the most open models out there. So this is a large language model that was trained by Meta and released open source about a year ago, in April 2024. This gives us a peek into how large-scale LLMs are actually trained these days. By the way, there is a new Llama4 model that just came out from Meta last month, April 2025.

There are slightly better models out there in open source already, but there's no paper on Llama4 yet. I'm excited to read that one hopefully when it comes out in a couple of months and see what can we learn from the new generation of Llama training. But just as a running example through today's lecture, we'll be pointing out a lot of examples from the Llama3-405B model for this reason.


<p align="center"><img src="./lecture_11_slides/slide_5206_00-02-53.707.jpg" width="75%" alt="Lecture Video at 00:02:53.707" /></p>

I want to talk about two things today. One is a bit about GPU hardware, and the other is how to train on lots of GPUs.


<p align="center"><img src="./lecture_11_slides/slide_5574_00-03-05.986.jpg" width="75%" alt="Lecture Video at 00:03:05.986" /></p>

<p align="center"><img src="./lecture_11_slides/slide_5644_00-03-08.322.jpg" width="75%" alt="Lecture Video at 00:03:08.322" /></p>

First, we're going to talk a little bit about GPU hardware. For those of you that don't know, GPU stands for Graphics Processing Unit. These were specialized coprocessors that were originally developed for computer graphics. They turned out to be very useful generalizable parallel processors.

It is actually very fitting to be giving this lecture in this room because this is the Huang Auditorium. Jen-Hsun Huang is the CEO and founder of NVIDIA, which is the biggest company right now and has been for the last couple of decades in producing GPUs, both for gaming and for ML. So these things started off basically for graphics. If you think about it, when you're doing computer graphics, you need to generate a lot of pixels on the screen.

You need to process lots of little pieces of primitive geometry to produce those pixels. It's very natural to do a lot of computation all in parallel when you're doing computer graphics. In the early days, in the early 2000s, researchers figured out how they could contort these graphics cards into doing generalizable parallel programming. They didn't quite know at the time what exactly they were going to be used for, I think.

I think they had this general idea that parallel processing was going to be important, and they really capitalized on deep learning when it started to take off in the early 2010s. It has basically been the main way that people train large-scale deep learning models for more than a decade now. That's starting to change, as we'll see a little bit, but their chips are the main one that people use.


<p align="center"><img src="./lecture_11_slides/slide_9102_00-05-03.704.jpg" width="75%" alt="Lecture Video at 00:05:03.704" /></p>

I always like looking inside these things and seeing what's in them. This is a picture of the NVIDIA H100, which is the mainstay of deep learning training right now. There's a next generation that just came out, but it's not really accessible yet. I haven't trained anything on it yet, so this is the state-of-the-art right now.

Inside this NVIDIA GPU, inside this H100 GPU in the middle here are these compute cores.


<p align="center"><img src="./lecture_11_slides/slide_9796_00-05-26.860.jpg" width="75%" alt="Lecture Video at 00:05:26.860" /></p>

And surrounding that are $80$ gigabytes of HBM memory (High Bandwidth Memory). You can see the memory is separated from the compute cores; they need to move and talk to each other over this bus to move data back and forth from the GPU memory into the cores. It can do that at a speed of about $3$ terabytes per second, which is a lot of bits moving around.


<p align="center"><img src="./lecture_11_slides/slide_10366_00-05-45.879.jpg" width="75%" alt="Lecture Video at 00:05:45.879" /></p>

<p align="center"><img src="./lecture_11_slides/slide_10438_00-05-48.281.jpg" width="75%" alt="Lecture Video at 00:05:48.281" /></p>

<p align="center"><img src="./lecture_11_slides/slide_10938_00-06-04.965.jpg" width="75%" alt="Lecture Video at 00:06:04.965" /></p>

The real heart of the thing are these $132$ Streaming Multiprocessors, or SMs. This is because all GPU hardware uses a process called binning. They plan for that in the development of their products. They say we're going to try to make a chip where the full chip theoretically has $144$ units, but none of the chips are perfect.

But they know they'll get a reasonable number of those that have at least $132$ that are functioning, so they use this process of binning. They then only sell a much larger proportion of the chips they tried to produce by only guaranteeing that $132$ of them will be turned on.


<p align="center"><img src="./lecture_11_slides/slide_13420_00-07-27.781.jpg" width="75%" alt="Lecture Video at 00:07:27.781" /></p>

We can dive even deeper inside one of those streaming multiprocessors and see more of what is going on inside these GPUs.


<p align="center"><img src="./lecture_11_slides/slide_13858_00-07-42.395.jpg" width="75%" alt="Lecture Video at 00:07:42.395" /></p>

This is just one of the $132$ active streaming multiprocessors inside an H100, and there are a couple interesting elements to look at. First, we see we have $256$ kilobytes of L1 cache and register files. This continues the trend of the memory hierarchy in the GPU. You may think you were learning deep learning, but you're actually learning computer architecture—sorry, it's a surprise.

Memory hierarchy is really important for deep learning and for all kinds of high-performance computing. The general trend is that you have larger bits of memory that are farther away from the compute cores. As you get closer to the compute cores, you have smaller bits of memory, but they are much, much faster. If you're writing performance GPU kernels, you spend a lot of time trying to optimize that.

To give you a flavor of that, we see the three levels of memory hierarchy in the H100: $256$ kilobytes of L1 cache, $50$ megabytes of L2 cache, and then $80$ gigabytes of HBM memory. These are the three primary levels of memory hierarchy in the H100.


<p align="center"><img src="./lecture_11_slides/slide_15794_00-08-46.994.jpg" width="75%" alt="Lecture Video at 00:08:46.994" /></p>

We also have $128$ FP32 cores. These are little arithmetic units that can do generalized floating-point operations. In particular, each one of these $128$ FP32 cores can compute $ax + b$, where $a$, $x$, and $b$ are all scalars. It can perform that bit of computation in one clock cycle.

If you add this up, the calculation $ax + b$ is basically $1$ multiply $1$ addition, and you have $128$ of these cores. This whole SM can do $256$ floating point operations per SM per clock cycle of the device.


<p align="center"><img src="./lecture_11_slides/slide_16856_00-09-22.429.jpg" width="75%" alt="Lecture Video at 00:09:22.429" /></p>

We also see that in red, there are four tensor cores—this is where the real magic happens. In addition to these FP32 cores, there are these tensor cores. I think the name is a little bit of a misnomer; these are actually matrix cores. What each of these little tensor cores does is they are special circuits designed to do only one thing: matrix multiply.

Each little tensor core can perform a single chunk of matrix multiply. In particular, the H100 ones can do a $16$-input matrix $A$ that is $16$ by $4$, input matrix $B$ that is $4$ by $8$, plus a bias matrix of size $16$ by $8$. It basically does $ax + b$, where $a$, $x$, and $b$ are little matrix chunks of this fixed size. It can do that one little chunk of matrix multiply once per tensor core per clock cycle.

We multiply that by the four tensor cores in the SM, and we see that the entire SM, if it's going through the tensor cores, can do 4,096 floating point operations per SM per clock cycle. This needs to be compared with the 256 that we can get from the $\text{FP}32$ cores. Here we see that just like the tensor cores are where all the magic happens; this is where the main throughput of the device comes from.

If you're writing code that wants to run on these GPUs and they can make maximum usage of them, you need to maximize usage of these tensor cores. Another interesting thing about these tensor cores is that they actually operate in mixed precision. Rather than traditional floating point numbers, which are normally 32-bit, the tensor cores tend to use a mixed precision procedure where the inputs are usually 16-bit.

There are a couple of different interesting 16-bit formats that they can use that we can't get into today. They perform the multiplications in this lower precision, 16-bit, and then do the additions—the accumulations—in a higher precision, 32-bit. So these tensor cores take a low precision, 16-bit input, perform some of the intermediate computation, and produce the outputs in a higher precision 32-bit.

This seems like a little bit of minutia, but it becomes very tangible when you mess up those data types in your PyTorch code.


<p align="center"><img src="./lecture_11_slides/slide_21446_00-11-55.582.jpg" width="75%" alt="Lecture Video at 00:11:55.582" /></p>

GPUs are really fast, and it is crazy how much faster they have gotten over the past decade or 15 years or so. When I first started my $\text{PhD}$ and was working on deep learning, the state-of-the-art GPU that we were all using was the K40 GPU, which was released back in 2013. This thing could do just a 5 teraflops of $\text{FP}32$ compute for the whole device.

You can see the graph goes up a lot. Something salient to notice here is that from the K40 to the P100, something really amazing happened in the V100, which came out towards the end of my $\text{PhD}$ and around 2016–2017. The V100 was the first device that introduced these tensor cores. The most recent device is the B200 that was formally announced and is slowly rolling out now.


<p align="center"><img src="./lecture_11_slides/slide_24234_00-13-28.607.jpg" width="75%" alt="Lecture Video at 00:13:28.607" /></p>

If you step back, this is like literally we've been living through a 1,000-fold increase in computation over the past 12 years, and that's just at the per device level. One explanation of why $\text{AI}$ has gotten so good in the last 10 years—what has happened? This is the answer: there's now a source of computation that we're taking advantage of, and it's gone up by $1,000$x in the last decade.

Anytime anything in the world changes by $1,000$x, you should step up and pay attention because that's going to cause major changes in our technological capabilities. This 1,000x improvement is the major driver of improvement in deep learning over the past decade. The device does not have 5,000 tensor cores; that is 5,000 teraflops of compute on the tensor cores.

We always try to distinguish between the compute on the tensor cores versus the compute on the $\text{FP}32$ cores. It is already crazy that there's been a 1,000 increase in a device that you can hold in your hands.


<p align="center"><img src="./lecture_11_slides/slide_26436_00-14-42.081.jpg" width="75%" alt="Lecture Video at 00:14:42.081" /></p>

That's insane. It gets even crazier because we don't train on one GPU. When the K40 first came out in 2013, it was common to train a lot of models on just one GPU. Stack that on top of this 1,000-fold increase in per device throughput, and something truly insane has happened in the past decade.


<p align="center"><img src="./lecture_11_slides/slide_27268_00-15-09.843.jpg" width="75%" alt="Lecture Video at 00:15:09.843" /></p>

We've looked inside the $\text{GPU}$. From here, I want to zoom out and put that $\text{GPU}$ in context. Not looking at individual devices, but thinking about the modern GPU clusters that we build that stitch a lot of these things together. So we've already seen a single H100 GPU.

We can think of it as another level of memory hierarchy. We saw inside the H100 there were three layers of memory hierarchy as we got closer to the compute elements. This trend actually continues once you escape the bounds of a single device and imagine these in the context of a full data center. Here we saw that a single H100 GPU gets about 3 terabytes of memory bandwidth.

That is the GPU memory talking from its own HBM memory to its own compute elements, 3 terabytes per second.


<p align="center"><img src="./lecture_11_slides/slide_28876_00-16-03.496.jpg" width="75%" alt="Lecture Video at 00:16:03.496" /></p>

It can move bits around, but these things typically live inside a GPU server. Almost all GPU servers have eight devices in one big box, and those GPUs can talk to each other. They typically talk to each other at a rate of about 900 gigabytes per second from any one GPU in the server to any other GPU in the server. You can see that is like a $3\times$ less memory communication bandwidth compared to the GPU talking inside one device.


<p align="center"><img src="./lecture_11_slides/slide_29674_00-16-30.123.jpg" width="75%" alt="Lecture Video at 00:16:30.123" /></p>

Here we again turn to Llama3. Some of the specifics may vary a little bit from cluster to cluster, but these are numbers from the Llama3 cluster that was used to train their models. They put two GPU boxes into one server rack. If you haven't seen it, a server rack is about 6 feet tall, like about the size of a person, to just get a mental picture of one of those things.

One server rack has two servers inside of it, totaling 16 GPUs.


<p align="center"><img src="./lecture_11_slides/slide_30796_00-17-07.560.jpg" width="75%" alt="Lecture Video at 00:17:07.560" /></p>

We connect a lot of server racks together into a GPU pod. The Llama3 cluster has GPU pods composed of 192 racks, which is a total of 3,072 GPUs. These things have really high bandwidth connectors between all the different racks. As a result, any pair of GPUs inside that pod can talk to each other at a rate of about 50 gigabytes per second.

This shows another $20\times$ decrease in memory traffic between what an individual server can talk and what any GPU across an entire rack can talk to each other. While 3,072 GPUs seems like a lot of compute, it is nowhere near enough.


<p align="center"><img src="./lecture_11_slides/slide_32014_00-17-48.201.jpg" width="75%" alt="Lecture Video at 00:17:48.201" /></p>

We are going to stack those GPU pods together into a full GPU cluster. This is the full GPU cluster that Meta built to train their Llama3 models; this thing combines eight GPU pods together for a total of 24,576 GPUs. I could not find exact numbers on the memory traffic between these things, but it's definitely less than 50 gigabytes per second. By the way, this is not the largest GPU cluster in the world by a long shot; it is the biggest one that I could quickly find precise numbers on.

But there are definitely GPU clusters out there in the world that are 50,000 GPUs or 100,000 GPUs. They exist, and people train models on them. The way that this works is it scales out naturally. How long do they train with that GPU cluster?

I don't remember offhand for the Llama3 models, but there has been kind of a rule of thumb for the past decade: the longest models that people train are usually on the order of months. That, I think, has less to do with technology and more to do with people. When it comes to having progress, making plans, and having people work on things, it is very difficult to have training runs that are very, very long.

The longest training runs for the biggest state-of-the-art models, I think, are typically measured in months. I would not be surprised if the very largest models—like GPT-4.5 or GPT-5—are pushing closer to a year at this point. But it is pretty common to see training runs that are on the order of a couple of months on these really big training clusters.

The question is, why do you organize servers into a rack rather than in a pod? You have to put them somewhere; there are physical constraints on these things. Server racks have been a standard unit in just data centers for decades at this point. When new devices like GPUs came onto the scene, that gave you a different kind of server.

They are physically bigger and they have a lot more power, but you cannot redesign the whole data center from scratch overnight. As a result, the server rack has been a standard unit with standard hardware sizes and everything that the data centers are typically built around. You should think of a single server rack as being like around 6 to 8 feet tall, something like that, about this big.

Maybe a server rack would be around the size of this podium and about as tall as me. Then you have 192 racks in a pod. Imagine 200 of these podiums; how big would that be? And then multiply that by 8.

But that's actually a little bit of an underestimate because you typically organize these things in rows so people can actually walk between them. There's more hardware that you need to pack into the cluster, not just the compute racks. So, in addition to the compute racks that have the physical GPU servers, there will be other racks that contain networking hardware.

We've got a lot of bits that need to fly around between all these devices, so they'll be dedicated racks that only hold networking hardware. There will also be dedicated racks that only hold storage hardware because you need to store that training data somewhere and get that into your devices. These things can take up quite a lot of space. Question is, when you go to these big clusters, do the smaller units of compute maintain the higher throughput?

Yes, they do. Oh, how hot does it get? Pretty hot. It will make the room physically warmer.

Although another interesting thing is about—I mean, the cooling gets crazy. A gaming desktop will typically be air cooled, sometimes water cooled, and then you can design different cooling systems. You can go nuts on the hardware here to try to optimize all this stuff. So I think this stuff is super cool.

It's just imagining like these GPUs are not just mythical creatures that are floating around in the cloud. These are actual physical atoms that someone built and stacked up in a room somewhere, and it's really interesting to imagine what they look like.


<p align="center"><img src="./lecture_11_slides/slide_39806_00-22-08.193.jpg" width="75%" alt="Lecture Video at 00:22:08.193" /></p>

Basically, one kind of mindset shift when we moved to these big GPU clusters is actually thinking not so much about the individual devices, about the individual servers. I basically tried to think of the entire data center as one big computer. This big computer in this case has $24,000$ GPUs, $1.8$ terabytes of HBM memory on the GPUs, $415$ million $\text{FP}32$ cores, and $13$ million tensor cores.

This whole thing can do $24$ exaflops of compute per second—that's $24 \times 10^{18}$. That's a lot of flops. It's a lot of flops, but I guarantee you five years from today it will not feel like a lot of flops, which is the even crazier part. Our goal here is actually to think of this entire block of $24,000$ GPUs as one giant supercomputer.

The question is, how can we train one neural network for months at a time on this one giant supercomputer? We need to train a really gigantic neural network that's really powerful, that can soak up tons and tons of data. That's basically the question and the paradigm that we've moved to in deep learning. By the way, I keep saying GPU; I keep saying NVIDIA because they are the most dominant training architecture and hardware today.


<p align="center"><img src="./lecture_11_slides/slide_41964_00-23-20.199.jpg" width="75%" alt="Lecture Video at 00:23:20.199" /></p>

But there are some others that have sprung up. The biggest competitor, I think right now to NVIDIA training hardware, is Google. Google has their own hardware called Tensor Processing Units (TPUs), and these are really good. They've gone through six generations of these already.

These are the stats of the v5p TPU, which you can rent in Google Cloud today. It's roughly same order of magnitude, similar specs as the H100 that we just talked about. There are some interesting design decisions in the TPU that are quite different from the GPUs, which I find fascinating, but we just don't have time to get into today. Someone was asking how big are these things?

This is an actual picture. Just like GPUs, these TPUs are arranged into pods, and the v5p TPUs can be arranged in pods of up to $8960$ chips. This is a picture, actually, of a V2 TPU pod, which has only $256$ chips. Then that gives you a sense of how big these things are, each one of those.

You see there's four racks here. Those racks are, like I said, maybe about a little bit taller than me. There are four of them side by side for $256$ TPU chips. And now imagine this thing is going to get a lot bigger in the more recent pods that have up to almost $9,000$ chips.

Yes, so Google's Gemini models are almost certainly trained on TPUs. Of course, they don't tell you, but I would be astounded, absolutely astounded if they were not. Like I said, the TPUs are actually very good. I assume that most large-scale Google models are trained on these things, and those are very competitive models.

So this is really good training hardware. The difference with NVIDIA is you can't buy it; the only way you can access TPUs are either by working at Google or by renting them on Google Cloud. But it is very good hardware, and a lot of people are making use of it, but I think it's still a little bit less popular today than NVIDIA GPUs. Of course, other companies obviously know that this is a very important thing.

So there are a lot of other companies that are trying to build competitive training hardware. But I think my honest assessment right now is that probably NVIDIA and TPUs are the two big ones. They're way ahead of everyone else right now, today, in terms of usability, performance, just like market share. But there are a lot of others that are trying to catch up here.


<p align="center"><img src="./lecture_11_slides/slide_45744_00-25-26.324.jpg" width="75%" alt="Lecture Video at 00:25:26.324" /></p>

Two notable ones are AMD. AMD has been the second major GPU manufacturer for many decades. They also have a training accelerator called the MI325X. On paper, it actually has really good stats that are pretty comparable to an H100, but it just hasn't had the same impact as the H100 right now.

AWS also has their own training chip that they've developed called Trainium. I don't know much about this one; I've never tried to use it myself, but I know that Anthropic uses it for some of their training. I don't know to what extent their training is entirely on Trainium versus GPUs, so we should expect to see more. But today, I think NVIDIA GPUs are probably the most dominant.

And Google TPUs are right there. They're really good as well but probably not quite as widely used as GPUs from NVIDIA.


<p align="center"><img src="./lecture_11_slides/slide_47154_00-26-13.372.jpg" width="75%" alt="Lecture Video at 00:26:13.372" /></p>

Ok. So that's basically part one: What are GPUs? How do we arrange them into clusters?


<p align="center"><img src="./lecture_11_slides/slide_47314_00-26-18.710.jpg" width="75%" alt="Lecture Video at 00:26:18.710" /></p>

This gives you a sense of the physicality of these machines that we're building and training on. Then the second question is, how do we actually write algorithms that can make use of this giant GPU cluster with tens of thousands of GPUs? It's going to require us to develop new algorithms, new ways of thinking about our compute, and new ways of parallelizing and splitting up our neural networks.

So the basic strategy here is going to be split up your computation. These things are giant parallel devices. They have a lot of—we saw they have a lot of GPUs, a lot of CPU cores, a lot of GPU cores. They can all operate independently, and they can't talk to each other too much.

A lot of this is specific to transformers because those are the dominant architecture that people are using for large scale training.


<p align="center"><img src="./lecture_11_slides/slide_50164_00-27-53.806.jpg" width="75%" alt="Lecture Video at 00:27:53.806" /></p>

If you think about a transformer, a transformer is basically a stack of $L$ layers. And each one of those $L$ layers is operating on a three-dimensional tensor of size, where one dimension is the minibatch dimension. We've got a bunch of sequences all operating in a minibatch, a sequence dimension. We're operating on sequences or sets of tokens, and a dim dimension.

So each of those tokens itself is a vector with some dimension. Our transformers are operating on these three-dimensional tensors, and they operate through a stack of layers. That gives us four axes to parallelize on. We can parallelize on the layers axis, which is pipeline parallelism.

We can parallelize on the batch dimension, which is data parallelism. We can split on the sequence dimension, which is called context parallelism. And we can split on that dim dimension, which is called tensor parallelism. We're going to step through each one of these in more detail because there's a lot of interesting nuances with all of these different mechanisms of distributed training.


<p align="center"><img src="./lecture_11_slides/slide_52174_00-29-00.872.jpg" width="75%" alt="Lecture Video at 00:29:00.872" /></p>

<p align="center"><img src="./lecture_11_slides/slide_52280_00-29-04.409.jpg" width="75%" alt="Lecture Video at 00:29:04.409" /></p>

The first one is Data Parallelism, or DP. And the basic idea here is simple. Remember, when we're training neural networks, we're always operating on minibatches of samples. We're always taking a minibatch of elements.

We're computing a loss for every entry in our minibatch, depending on whatever our training task is. Then we compute a gradient, where the gradient is actually typically an average of the gradients of the losses for the individual elements in the minibatch. So in most neural network architectures, the computation of the loss and then computing the gradient is independent for each of the elements in the minibatch.


<p align="center"><img src="./lecture_11_slides/slide_53268_00-29-37.376.jpg" width="75%" alt="Lecture Video at 00:29:37.376" /></p>

This is something that seems trivially parallelizable.


<p align="center"><img src="./lecture_11_slides/slide_53882_00-29-57.863.jpg" width="75%" alt="Lecture Video at 00:29:57.863" /></p>

If you think about mathematically why this makes sense, it's because gradients are linear. And then the $W$ are the weight matrices of the entire network. Then typically the loss that you're computing at the end of the forward pass is an average of the losses on each of the individual minibatch elements.

And then if you take the gradient of the loss with respect to the weights of the network, that's the thing we need to compute in order to make a weight update. Then that is actually going to split because gradients are linear; you get to choose what order do we want to do the sum? Do we want to do the gradient? Do we want to do the averaging?


<p align="center"><img src="./lecture_11_slides/slide_55320_00-30-45.844.jpg" width="75%" alt="Lecture Video at 00:30:45.844" /></p>

And these can be computed in parallel on different GPUs.


<p align="center"><img src="./lecture_11_slides/slide_55726_00-30-59.391.jpg" width="75%" alt="Lecture Video at 00:30:59.391" /></p>

And then there's an outer sum where we need to take an average of the gradients across our $M$ different devices that we're operating on. So that's what's happening from a mathematical perspective. We see that this is perfectly mathematically sound. This is basically exactly the same mathematically as training on a single device.

We've just been clever with our algebra and changed the order of doing our averages and our summations.


<p align="center"><img src="./lecture_11_slides/slide_56586_00-31-28.086.jpg" width="75%" alt="Lecture Video at 00:31:28.086" /></p>

But this is not an approximation; it is exactly the same computation as we would have done on a single larger GPU. So what this looks like at the GPU perspective is that we have $M$ GPUs. Here I'm showing $M = 3$ because that's all that can sensibly fit on the slide, but think that this is much larger than 3 in practice. Each one of those GPUs actually maintains its own separate copy of the neural network weights, of the optimizer state, and of the gradients.


<p align="center"><img src="./lecture_11_slides/slide_57236_00-31-49.775.jpg" width="75%" alt="Lecture Video at 00:31:49.775" /></p>

Then what we're going to do: each GPU will load in parallel a different minibatch of data. I've had bugs in my code and students' code where they actually accidentally load the same minibatch on all the GPUs. That's not going to help you; that's not going to be good. Don't make that mistake.

So it's crucially important that your different GPUs actually load different minibatches of data.


<p align="center"><img src="./lecture_11_slides/slide_58098_00-32-18.537.jpg" width="75%" alt="Lecture Video at 00:32:18.537" /></p>

Then each GPU will independently do its own forward pass on its own minibatch of data to compute its own local loss on its own local minibatch of data.


<p align="center"><img src="./lecture_11_slides/slide_58472_00-32-31.016.jpg" width="75%" alt="Lecture Video at 00:32:31.016" /></p>

These can all operate totally independently; it does not require any communication between GPUs. Next, each network will do its own backward pass to compute the gradient of its own local loss with respect to all the weights of the model. And again, this can happen totally independently because each model—remember, each GPU has its own independent copy of the model weights.

It can do its own forward-backward pass completely independently. But now after the backward pass is done, this is where things get tricky. Remember, we said we needed to compute an average of those gradients across all the devices that are participating in our training. So then we need communication.


<p align="center"><img src="./lecture_11_slides/slide_59468_00-33-04.249.jpg" width="75%" alt="Lecture Video at 00:33:04.249" /></p>

This is where we do an All-Reduce operation. Every GPU needs to send its gradients to all the other GPUs. Two things are happening simultaneously. One, each GPU needs to broadcast its gradients to all the GPUs.

And then two, each GPU needs to collect the gradients from all the GPUs that are participating in the training. This is an All-Reduce operation. And this happens in logarithmic time, typically depending on the number of GPUs. But at the end of this All-Reduce operation, each GPU now has an average of all the gradients across all the devices.

At this point, the communication has happened. Each GPU now has an identical copy of the gradients that have been all reduced across all the devices. So now, at the beginning of the training iteration, we assumed that each GPU had its own independent copy of the model weights. Now, at this point, each GPU has its own independent but identical copy of the gradients across the entire macro batch of data.


<p align="center"><img src="./lecture_11_slides/slide_61200_00-34-02.040.jpg" width="75%" alt="Lecture Video at 00:34:02.040" /></p>

At this point, each GPU can make a weight update on its own local copy of the weights.


<p align="center"><img src="./lecture_11_slides/slide_61612_00-34-15.786.jpg" width="75%" alt="Lecture Video at 00:34:15.786" /></p>

Also, by the way, this is really important. Steps 4 and 5 can actually happen in parallel. In practice, these things will typically happen simultaneously. That means that each model will start off doing a backward pass over the last layer in the network and then compute its own local gradient.

The model will move its compute onto computing a backward pass for the second-to-last layer of the model. And while the compute elements are busy computing the backward pass on the second-to-last layer, the GPUs will simultaneously be doing an All-Reduce of the gradients of the last layer. This means that these things chunk along communication for layer $L+1$ and backward pass for layer $L$.

We can make our weight update all at once without waiting. This is really important because like we said, the communication is relatively slow. So the whole trick in these things is figuring out ways to hide the communication costs and do them at the same time as the compute. The question is, is four or five going to be the bottleneck?

And the answer is yes. It depends entirely on how fast your device is. How big is your model? How big is your minibatch?

How fast is the interconnect between the devices? When you get to this large-scale distributed training, the answer is always it depends on your situation, and you need to benchmark for your situation. Why not take $M$ different gradient steps on each of them? That's actually a really cool idea.

Those were popular; Google used to do this before they developed the TPU pods, and some of their earlier networks in the early 2010s were trained in this way. But one, it tends to just be a lot more unstable, and two, it's very hard to debug and reproduce. And it just tends to work a little bit worse, so it does feel like a more scalable approach. But in practice, if you can do everything synchronously, then your algorithms are easier to debug, easier to understand, easier to reason about.

Basically, if you can get away with synchronous gradient updates, it's probably going to work better. There's no one computer that can orchestrate all this stuff. All these things are independent devices with their own independent stuff. There's no driver that can take a God's eye view and take those steps; all that computation has to happen somewhere.

I said, as you're overlapping communication and compute, do you need to write code for this, or does the hardware do this automatically? You definitely got to write code for this. The hardware is not smart enough to understand what you want to do. The hardware, like we said, it understands these little matrix multiply chunks; it understands pretty low-level stuff.

Anything that you want to do to schedule that communication, you need to take care of in software. But thankfully for a lot of these common use cases, PyTorch ships with it for you. But at the cluster level, it typically doesn't, so then you typically need to do it in software. Typically these are heterogeneous systems where different parts of the system are written in different programming languages.

But then those individual GPU kernels will get wrapped up, and you can call those GPU kernels from Python.


<p align="center"><img src="./lecture_11_slides/slide_69602_00-38-42.386.jpg" width="75%" alt="Lecture Video at 00:38:42.386" /></p>

<p align="center"><img src="./lecture_11_slides/slide_69624_00-38-43.121.jpg" width="75%" alt="Lecture Video at 00:38:43.121" /></p>

In this picture, each GPU is computing its own gradients (in black) by itself, and then the gradients in red get computed via an allreduce across all the GPUs in parallel. The backward pass at a lower layer is dependent on the gradients from the previous layer. But crucially, each GPU is only doing the backward pass locally on its own minibatch. In order to compute a backward pass, each GPU only needs the local version of its upstream gradient, but then computing the global version of the upstream gradient requires communication.


<p align="center"><img src="./lecture_11_slides/slide_71128_00-39-33.304.jpg" width="75%" alt="Lecture Video at 00:39:33.304" /></p>

So this is data parallelism. But we quickly hit a bottleneck on the model size. If we're using Adam, that's typically $\beta_1$ and a $\beta_2$ per parameter in the network, and sometimes you'll also have an exponential moving average of the model parameters as well. So typically, you'll have four to five scalars that you need to keep track of for every weight in your network.

If you're training with 16-bit precision, which is pretty common these days, some of these you'll sometimes keep in higher precision. But let's talk about 16-bit as a lower bound: then you need two bytes for each number. That means that we need four numbers. That's not big enough; we want really big models.

We don't want to be constrained by the tyranny of our GPU memory size in telling us how big of models we're allowed to train, so we need to fix this somehow.


<p align="center"><img src="./lecture_11_slides/slide_73906_00-41-05.997.jpg" width="75%" alt="Lecture Video at 00:41:05.997" /></p>

<p align="center"><img src="./lecture_11_slides/slide_74192_00-41-15.540.jpg" width="75%" alt="Lecture Video at 00:41:15.540" /></p>

The fix for this is actually relatively easy: we need to split the model weights across the different GPUs. In addition to splitting the batch of data across GPUs, we are also going to split our model weights across the GPUs. This leads to a variant of data parallelism called Fully Sharded Data Parallelism, or FSDP, which is conceptually simple. Conceptually, each model weight in the network, each weight $W_i$, gets assigned to an owner GPU.

Every weight will be owned by a unique GPU among the $M$ GPUs that we're training on.


<p align="center"><img src="./lecture_11_slides/slide_76816_00-42-43.094.jpg" width="75%" alt="Lecture Video at 00:42:43.094" /></p>

<p align="center"><img src="./lecture_11_slides/slide_77546_00-43-07.452.jpg" width="75%" alt="Lecture Video at 00:43:07.452" /></p>

<p align="center"><img src="./lecture_11_slides/slide_78496_00-43-39.150.jpg" width="75%" alt="Lecture Video at 00:43:39.150" /></p>

The GPU that owns each weight is also responsible for managing the global gradients and the optimizer state for that weight. Typically, you would split this up by layer; you are not managing individual scalars here. This $W$ should be thought of like the weight matrix for an entire layer of the neural network. We are showing a four-layer network being distributed across two different GPUs.

We've assigned the weights for the first two network layers, $W_1$ and $W_2$, which are owned by GPU 1. The weights $W_3$ and $W_4$ are owned by GPU 2. At the start of each batch, the network weights are split up across the GPUs in this way. This algorithm gets tricky now because the model weights are split up, requiring us to introduce extra communication.

For example, if GPU 1 owns $W_1$, it broadcasts that to GPU 2. Now that all the GPUs have a copy of $W_1$, they can run a forward pass through the first layer and compute the activations there. After running the forward pass, each GPU that does not own $W_1$ deletes its local copy of the weight matrix to save memory. After this, we are back in the state where model weights are split up across the GPUs, but all GPUs also have an activation stored in GPU memory from the first layer.

It is now time for the second layer, and we repeat the process: the GPU that owns the weight matrix for layer 2 broadcasts it to all GPUs.


<p align="center"><img src="./lecture_11_slides/slide_78762_00-43-48.026.jpg" width="75%" alt="Lecture Video at 00:43:48.026" /></p>

<p align="center"><img src="./lecture_11_slides/slide_78796_00-43-49.160.jpg" width="75%" alt="Lecture Video at 00:43:49.160" /></p>

Now they all have their own local copy of $W_2$ and can proceed forward.


<p align="center"><img src="./lecture_11_slides/slide_79192_00-44-02.374.jpg" width="75%" alt="Lecture Video at 00:44:02.374" /></p>

In practice, this happens in parallel during the forward pass of an FSDP run. We will be computing layer 2 while simultaneously fetching the weights for layer 3. Once we get to layer 3, if GPU 1 owns it, GPU 1 broadcasts the weights to all GPUs being trained on.


<p align="center"><img src="./lecture_11_slides/slide_79728_00-44-20.258.jpg" width="75%" alt="Lecture Video at 00:44:20.258" /></p>

<p align="center"><img src="./lecture_11_slides/slide_79758_00-44-21.259.jpg" width="75%" alt="Lecture Video at 00:44:21.259" /></p>

This repeats until we reach the end of the network. At the end of the network, every model has done a full forward pass, computed its local loss on its own local batch, and has all activations for all layers in memory already for the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_80138_00-44-33.938.jpg" width="75%" alt="Lecture Video at 00:44:33.938" /></p>

We now need to do the same thing in reverse to compute the backward pass. At the beginning of the backward pass for the last layer, whoever owns that weight will broadcast it to all devices. Once the devices have that weight, they can perform the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_80700_00-44-52.690.jpg" width="75%" alt="Lecture Video at 00:44:52.690" /></p>

This whole procedure is similar in the backward pass. There is a small optimization on the very last layer: having all GPUs keep the weights for the last layer in memory.


<p align="center"><img src="./lecture_11_slides/slide_81428_00-45-16.980.jpg" width="75%" alt="Lecture Video at 00:45:16.980" /></p>

<p align="center"><img src="./lecture_11_slides/slide_81450_00-45-17.715.jpg" width="75%" alt="Lecture Video at 00:45:17.715" /></p>

<p align="center"><img src="./lecture_11_slides/slide_81566_00-45-21.585.jpg" width="75%" alt="Lecture Video at 00:45:21.585" /></p>

<p align="center"><img src="./lecture_11_slides/slide_82066_00-45-38.269.jpg" width="75%" alt="Lecture Video at 00:45:38.269" /></p>

Then we need to communicate those gradients back. We said that the GPU that owns the weight matrix is also going to be responsible for managing the gradients for that weight matrix. What happens during the downtime? You got to get all this stuff happening in parallel.


<p align="center"><img src="./lecture_11_slides/slide_84300_00-46-52.810.jpg" width="75%" alt="Lecture Video at 00:46:52.810" /></p>

<p align="center"><img src="./lecture_11_slides/slide_84990_00-47-15.833.jpg" width="75%" alt="Lecture Video at 00:47:15.833" /></p>

<p align="center"><img src="./lecture_11_slides/slide_85086_00-47-19.036.jpg" width="75%" alt="Lecture Video at 00:47:19.036" /></p>

<p align="center"><img src="./lecture_11_slides/slide_85106_00-47-19.703.jpg" width="75%" alt="Lecture Video at 00:47:19.703" /></p>

<p align="center"><img src="./lecture_11_slides/slide_85176_00-47-22.039.jpg" width="75%" alt="Lecture Video at 00:47:22.039" /></p>

So there's basically three things that need to happen during backward. During backward, we need to communicate the weights. Whatever GPU owns the layer, owns the weights for that layer has to broadcast them. Two, we need all the GPUs once they get that weight, you need to compute a backward pass for that layer.

And then three, after each GPU computes its backward pass, it needs to send the result of the gradients with respect to the weights of that backward pass back to the $\text{GPU}$ that owns it. After that, once the owner of the weights has that full gradient, only the owner of the weight matrix can now make a gradient update on that one weight matrix.

But I think at this point, we actually do not need to communicate the updated weight matrix because it will get re-communicated to all the GPUs on the next forward pass. So that's a little bit different from the DP case, maybe. And then basically all of these things can actually happen in parallel as well. We'll repeat this for every layer of the network.

In the steady state of a very deep network, all three of these things will be happening simultaneously. So while we are computing the backward pass for layer $L$, we will be aggregating the gradients and performing a weight update on layer $L+1$. And we will be prefetching the weights for layer $L-1$. I said there's three things that need to happen: we need to get the weight, run the backward pass, and then aggregate the gradient and update the weight.

These things can all happen in parallel. So basically in general, we'll be operating on three consecutive layers and doing all three of these things in parallel over the course of the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_86350_00-48-01.212.jpg" width="75%" alt="Lecture Video at 00:48:01.212" /></p>

<p align="center"><img src="./lecture_11_slides/slide_86370_00-48-01.879.jpg" width="75%" alt="Lecture Video at 00:48:01.879" /></p>

<p align="center"><img src="./lecture_11_slides/slide_86384_00-48-02.346.jpg" width="75%" alt="Lecture Video at 00:48:02.346" /></p>

<p align="center"><img src="./lecture_11_slides/slide_86398_00-48-02.813.jpg" width="75%" alt="Lecture Video at 00:48:02.813" /></p>

All the GPUs have already finished doing their update on all the weights, and we're ready. Also hopefully your data loader that's loading data is also happening asynchronously, usually on the $\text{CPU}$ cores of our servers. So then the $\text{CPU}$ is ready with a fresh batch of data to go forward again. These things are basically parallelization machines.


<p align="center"><img src="./lecture_11_slides/slide_87682_00-48-45.656.jpg" width="75%" alt="Lecture Video at 00:48:45.656" /></p>

<p align="center"><img src="./lecture_11_slides/slide_87840_00-48-50.928.jpg" width="75%" alt="Lecture Video at 00:48:50.928" /></p>

Then we're basically ready to do our next batch after that.


<p align="center"><img src="./lecture_11_slides/slide_87864_00-48-51.729.jpg" width="75%" alt="Lecture Video at 00:48:51.729" /></p>

So this is great. This is Fully Sharded Data Parallelism. And this can get you a long way.


<p align="center"><img src="./lecture_11_slides/slide_88118_00-49-00.203.jpg" width="75%" alt="Lecture Video at 00:49:00.203" /></p>

But there's actually a slightly fancier variant of data parallelism that people sometimes use called Hybrid Sharded Data Parallelism, or $\text{HSDP}$. In this case, we're actually going to imagine conceptually dividing our $\text{GPUs}$ into a two-dimensional grid. In the previous examples, we said we had $N$ $\text{GPUs}$, and the way that we parallelized our computation was the same.

We had one axis of parallelization in the previous variants of data parallelism. Once we get to Hybrid Sharded Data Parallelism, we now are going to have two separate axes of parallelism that we will do at the same time. The first axis is we will do typical $\text{FSDP}$, Fully Sharded Data Parallelism, along one axis that we just talked about. We'll have groups of $K$ $\text{GPUs}$, and each group of $K$ $\text{GPUs}$ will be doing Fully Sharded Data Parallelism that we just talked about.

Within each group of $K$ $\text{GPUs}$, the model weights will be split across those $K$ $\text{GPUs}$. They will be interleaving, sending weights and gradients back and forth to each other during the forward and backward passes. But we will have now $M$ copies of those $K$ groups operating in parallel. In this case, we have two groups of four $\text{GPUs}$.

So each group of four $\text{GPUs}$ you see has the weights split across the four $\text{GPUs}$. But we have the entire setup duplicated a second time on a second group of two $\text{GPUs}$. When you do this, they do typical data parallelism across the groups. Within a group, we're going to do forward/backward.

At the end of the backward, each group will have computed its own local gradients. And then each group can make a gradient update independently once they've received the full gradients for the macrobatch.


<p align="center"><img src="./lecture_11_slides/slide_91376_00-50-48.913.jpg" width="75%" alt="Lecture Video at 00:50:48.913" /></p>

<p align="center"><img src="./lecture_11_slides/slide_91686_00-50-59.257.jpg" width="75%" alt="Lecture Video at 00:50:59.257" /></p>

This might be useful because there are different amounts of communication required for these two different kinds of parallelism. If we think about fully sharded parallelism (FSDP), what do we need to communicate during fully sharded data parallelism? During the forward pass, remember we were copying the weights all over. During the forward pass, we end up doing a communication of one full copy of the network weights.

Then during the backward pass, we need to re-communicate the network weights, and we also need to communicate the gradients. But when you do normal data parallelism, where each group keeps its own independent copy of the weights, you only need to all reduce the gradients. This means that across multiple data parallelism groups, you only need to communicate the network weights once over a forward-backward pass.

This plays into the idea of multiple levels of hierarchy inside of our GPU clusters. For example, what you might do is have a GPU server with eight GPUs and high interconnect inside a single machine. These might be an FSDP group because it requires more communication inside an FSDP group. You could then have multiple servers on this other axis.

So you have one server with a full copy of the model weights, and another server with another full copy of the model weights. Remember, communication across servers is going to be slower than communication inside a server. This is our first example of designing algorithms to take advantage of the network topology that we know our devices are connected into.


<p align="center"><img src="./lecture_11_slides/slide_94734_00-52-40.957.jpg" width="75%" alt="Lecture Video at 00:52:40.957" /></p>

But once you have data parallelism—once you have this DP, FSDP, and HSDP—this is actually a recipe that can take you a long way. For example, a model with $100$ billion parameters would take $800\,\text{GB}$ of memory to store. If you split that over $80\,\text{GPUs}$, it only takes $10\,\text{gigabytes}$ of memory per GPU. You can have a pretty big model once you have FSDP.


<p align="center"><img src="./lecture_11_slides/slide_95462_00-53-05.248.jpg" width="75%" alt="Lecture Video at 00:53:05.248" /></p>

However, there is another problem: the model activations themselves start to fill up memory. If we go back to Llama-3-$405\text{B}$, it's a transformer with $126$ layers, model dimension of $16,000$, and sequence length $4,096$. If you imagine how much GPU memory it takes just to store the hidden states during the forward pass, that is going to be a lot.


<p align="center"><img src="./lecture_11_slides/slide_96154_00-53-28.339.jpg" width="75%" alt="Lecture Video at 00:53:28.339" /></p>

This quickly causes your GPU to run out of memory once your models and sequences get really big.


<p align="center"><img src="./lecture_11_slides/slide_96420_00-53-37.214.jpg" width="75%" alt="Lecture Video at 00:53:37.214" /></p>

We're going to recompute them during the backward pass. To see how this works, it is useful to think of a neural network as having layers.


<p align="center"><img src="./lecture_11_slides/slide_97006_00-53-56.767.jpg" width="75%" alt="Lecture Video at 00:53:56.767" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97168_00-54-02.173.jpg" width="75%" alt="Lecture Video at 00:54:02.173" /></p>

If we assume all constants are the same, a typical forward-backward pass takes multiple steps during the forward pass where you remember those activations. Then there are more steps during the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_97256_00-54-05.109.jpg" width="75%" alt="Lecture Video at 00:54:05.109" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97270_00-54-05.576.jpg" width="75%" alt="Lecture Video at 00:54:05.576" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97284_00-54-06.043.jpg" width="75%" alt="Lecture Video at 00:54:06.043" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97420_00-54-10.581.jpg" width="75%" alt="Lecture Video at 00:54:10.581" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97448_00-54-11.515.jpg" width="75%" alt="Lecture Video at 00:54:11.515" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97462_00-54-11.982.jpg" width="75%" alt="Lecture Video at 00:54:11.982" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97474_00-54-12.383.jpg" width="75%" alt="Lecture Video at 00:54:12.383" /></p>

<p align="center"><img src="./lecture_11_slides/slide_97588_00-54-16.187.jpg" width="75%" alt="Lecture Video at 00:54:16.187" /></p>

In a normal forward-backward pass, it takes $\mathcal{O}(N)$ compute and $\mathcal{O}(N)$ memory for an $N$-layer network.


<p align="center"><img src="./lecture_11_slides/slide_97788_00-54-22.860.jpg" width="75%" alt="Lecture Video at 00:54:22.860" /></p>

But as we just said, this is going to run out of memory.


<p align="center"><img src="./lecture_11_slides/slide_97956_00-54-28.466.jpg" width="75%" alt="Lecture Video at 00:54:28.466" /></p>

Instead, what we can do is imagine recomputing the activations during the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_98032_00-54-31.001.jpg" width="75%" alt="Lecture Video at 00:54:31.001" /></p>

We start with the first layer, run the forward pass, and then immediately throw away those activations.


<p align="center"><img src="./lecture_11_slides/slide_98258_00-54-38.542.jpg" width="75%" alt="Lecture Video at 00:54:38.542" /></p>

We repeat this process for all layers.


<p align="center"><img src="./lecture_11_slides/slide_98274_00-54-39.076.jpg" width="75%" alt="Lecture Video at 00:54:39.076" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98396_00-54-43.147.jpg" width="75%" alt="Lecture Video at 00:54:43.147" /></p>

Now we've gone through the network once and got the activations at the last layer.


<p align="center"><img src="./lecture_11_slides/slide_98670_00-54-52.289.jpg" width="75%" alt="Lecture Video at 00:54:52.289" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98696_00-54-53.157.jpg" width="75%" alt="Lecture Video at 00:54:53.157" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98708_00-54-53.557.jpg" width="75%" alt="Lecture Video at 00:54:53.557" /></p>

We recompute them, run the backward pass, then recompute some more, and do another backward pass. If you add this all up, it ends up being $\mathcal{O}(N^2)$ compute and constant memory for a network with $N$ layers, because it sums from $1$ to $N-1$.


<p align="center"><img src="./lecture_11_slides/slide_98758_00-54-55.226.jpg" width="75%" alt="Lecture Video at 00:54:55.226" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98800_00-54-56.627.jpg" width="75%" alt="Lecture Video at 00:54:56.627" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98810_00-54-56.961.jpg" width="75%" alt="Lecture Video at 00:54:56.961" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98842_00-54-58.028.jpg" width="75%" alt="Lecture Video at 00:54:58.028" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98870_00-54-58.963.jpg" width="75%" alt="Lecture Video at 00:54:58.963" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98896_00-54-59.830.jpg" width="75%" alt="Lecture Video at 00:54:59.830" /></p>

<p align="center"><img src="./lecture_11_slides/slide_98994_00-55-03.100.jpg" width="75%" alt="Lecture Video at 00:55:03.100" /></p>

<p align="center"><img src="./lecture_11_slides/slide_99340_00-55-14.645.jpg" width="75%" alt="Lecture Video at 00:55:14.645" /></p>

That is quadratic time.


<p align="center"><img src="./lecture_11_slides/slide_99458_00-55-18.582.jpg" width="75%" alt="Lecture Video at 00:55:18.582" /></p>

Since $\mathcal{O}(N^2)$ compute is pretty bad for deep networks, we can instead imagine taking a checkpoint of activations every $C$ layers. We only recompute within tinier blocks of the network.


<p align="center"><img src="./lecture_11_slides/slide_99812_00-55-30.394.jpg" width="75%" alt="Lecture Video at 00:55:30.394" /></p>

<p align="center"><img src="./lecture_11_slides/slide_100138_00-55-41.271.jpg" width="75%" alt="Lecture Video at 00:55:41.271" /></p>

A pretty common thing to do is to set $C$ equal to $\sqrt{N}$, in which case This becomes $O(\sqrt{N})$ compute and $O(\sqrt{N})$ memory. So this is a pretty common way that you can trade-off computation and memory to train even bigger models. So now, at this point, once we have FSDP, activation checkpointing, HSDP, we can do a lot of damage here.


<p align="center"><img src="./lecture_11_slides/slide_100732_00-56-01.091.jpg" width="75%" alt="Lecture Video at 00:56:01.091" /></p>

<p align="center"><img src="./lecture_11_slides/slide_100818_00-56-03.960.jpg" width="75%" alt="Lecture Video at 00:56:03.960" /></p>

We can start to train some really big models.


<p align="center"><img src="./lecture_11_slides/slide_100858_00-56-05.295.jpg" width="75%" alt="Lecture Video at 00:56:05.295" /></p>

The recipe for that is basically as follows.


<p align="center"><img src="./lecture_11_slides/slide_100974_00-56-09.166.jpg" width="75%" alt="Lecture Video at 00:56:09.166" /></p>

You can just do normal data parallelism for models of this size; it tends to work pretty well.


<p align="center"><img src="./lecture_11_slides/slide_101366_00-56-22.245.jpg" width="75%" alt="Lecture Video at 00:56:22.245" /></p>

Another thing that you almost always want to set the local batch size per GPU to max out the GPU memory.


<p align="center"><img src="./lecture_11_slides/slide_101606_00-56-30.253.jpg" width="75%" alt="Lecture Video at 00:56:30.253" /></p>

That's almost always the right thing to do. It depends on how much memory your GPU has and how fast your interconnects are.


<p align="center"><img src="./lecture_11_slides/slide_102278_00-56-52.676.jpg" width="75%" alt="Lecture Video at 00:56:52.676" /></p>

At this point, you can scale up quite a bit, but then you'll run into the memory bottleneck for your activations. That's when you turn on activation checkpointing. Activation checkpointing makes everything a lot slower, but it does let you train much bigger models.


<p align="center"><img src="./lecture_11_slides/slide_102692_00-57-06.490.jpg" width="75%" alt="Lecture Video at 00:57:06.490" /></p>

This will scale up to several hundred GPUs. You need to start switching to HSDP. This basically going to let you get up to models that are roughly tens of billions of parameters, training on maybe a thousand GPUs on pretty long sequence lengths.


<p align="center"><img src="./lecture_11_slides/slide_103640_00-57-38.121.jpg" width="75%" alt="Lecture Video at 00:57:38.121" /></p>

So that's pretty good.


<p align="center"><img src="./lecture_11_slides/slide_104020_00-57-50.800.jpg" width="75%" alt="Lecture Video at 00:57:50.800" /></p>

There's a big question that comes up: there are a lot of knobs to tune here. How am I supposed to optimize this? I need to set the global batch size, the local batch size, the HSDP dimension, the FSDP dimension, how much to recompute.


<p align="center"><img src="./lecture_11_slides/slide_104522_00-58-07.550.jpg" width="75%" alt="Lecture Video at 00:58:07.550" /></p>

There are so many knobs; what do I do? The answer is to optimize a very important metric called Model Flops Utilization (MFU). Whenever you get lost in the sea of GPU parallelism, Model Flops Utilization is your guiding light.


<p align="center"><img src="./lecture_11_slides/slide_104938_00-58-21.431.jpg" width="75%" alt="Lecture Video at 00:58:21.431" /></p>

Follow this, and it will tell you what to do to optimize your training stack. But before we get to Model Flops Utilization, we need to talk about Hardware Flops Utilization. Remember we said that, in theory, an H100 can do $989.4$ TFLOPs per second of compute on the tensor cores, but that's theoretical.


<p align="center"><img src="./lecture_11_slides/slide_105476_00-58-39.383.jpg" width="75%" alt="Lecture Video at 00:58:39.383" /></p>

The question is: how much can you actually achieve in practice? That's the metric of Hardware Flops Utilization.


<p align="center"><img src="./lecture_11_slides/slide_105748_00-58-48.458.jpg" width="75%" alt="Lecture Video at 00:58:48.458" /></p>

<p align="center"><img src="./lecture_11_slides/slide_105966_00-58-55.732.jpg" width="75%" alt="Lecture Video at 00:58:55.732" /></p>

<p align="center"><img src="./lecture_11_slides/slide_107094_00-59-33.370.jpg" width="75%" alt="Lecture Video at 00:59:33.370" /></p>

<p align="center"><img src="./lecture_11_slides/slide_107670_00-59-52.589.jpg" width="75%" alt="Lecture Video at 00:59:52.589" /></p>

<p align="center"><img src="./lecture_11_slides/slide_108068_01-00-05.869.jpg" width="75%" alt="Lecture Video at 01:00:05.869" /></p>

<p align="center"><img src="./lecture_11_slides/slide_108476_01-00-19.483.jpg" width="75%" alt="Lecture Video at 01:00:19.483" /></p>

You're running some compute on the device; how much compute do you actually realize of that theoretical maximum? This is not hard to do. You can write a couple lines of PyTorch code and just benchmark this. This is a benchmark that I wrote that I ran on an H100 yesterday.

What it does is basically, the $x$-axis: it just does dense matrix multiply in a loop, and then times how long did the matrix multiply happen? We can compute how many flops a matrix multiply takes. On the $x$-axis, we're plotting the size of our matrix going from 512 up to 32,000. You can see that on this pretty straightforward PyTorch loop, we're getting about $80\%$ HFU on an H100 once we get to large matrix multiplies of around $8,000$ by $8,000$.

So that's pretty good. But the problem is that HFU does not account for all the other stuff that your model needs to do. There's a lot of other stuff your GPU is doing other than just forward and backward on your raw model. That's where we move from Hardware Flops Utilization to Model Flops Utilization.

So, Model Flops Utilization is basically saying what fraction of the GPU's theoretical TFLOPs are being used for forward and backward in my model? This is the thing you always want to optimize for. Then you look up somewhere the peak theoretical throughput of the device you're running on. And then you divide those two.


<p align="center"><img src="./lecture_11_slides/slide_108978_01-00-36.233.jpg" width="75%" alt="Lecture Video at 01:00:36.233" /></p>

<p align="center"><img src="./lecture_11_slides/slide_109014_01-00-37.434.jpg" width="75%" alt="Lecture Video at 01:00:37.434" /></p>

You then actually time a forward-backward pass of your model. Your training loop is doing all this other stuff; it's doing data loading, it's doing augmentation, it's doing communication. It's doing activation checkpointing. So it's doing recomputation, doing backward.

Your training loop is doing a lot of stuff. Just times see how long it actually takes, and then divide those two numbers. That gives you a number between 0 and 1, which is like what fraction of that theoretical maximum are you actually achieving in your training loop?


<p align="center"><img src="./lecture_11_slides/slide_109774_01-01-02.793.jpg" width="75%" alt="Lecture Video at 01:01:02.793" /></p>

<p align="center"><img src="./lecture_11_slides/slide_109898_01-01-06.930.jpg" width="75%" alt="Lecture Video at 01:01:06.930" /></p>

That's your MFU, your Model Flops Utilization. And again, we can benchmark this with some relatively simple PyTorch code.


<p align="center"><img src="./lecture_11_slides/slide_110456_01-01-25.549.jpg" width="75%" alt="Lecture Video at 01:01:25.549" /></p>

This is getting around 50% MFU.


<p align="center"><img src="./lecture_11_slides/slide_110970_01-01-42.699.jpg" width="75%" alt="Lecture Video at 01:01:42.699" /></p>

In general, an MFU these days generally above 30% is pretty good. If you're way under 30%, you've probably got some gigantic bottleneck somewhere and something is going wrong. And above 40% is pretty, pretty excellent, and that's basically state of the art.


<p align="center"><img src="./lecture_11_slides/slide_111316_01-01-54.244.jpg" width="75%" alt="Lecture Video at 01:01:54.244" /></p>

Here's some numbers that we can pull from a couple of papers. In particular, this is that Llama3-405B paper that we talked about. In their final training step, they have a couple different variants of their training phases where they train on between 8,000 and 16,000 GPUs simultaneously. Across that, they're getting MFUs roughly in the high 30s, low 40s.

And that's pretty good; you're never going to get really much higher than that on an H100.


<p align="center"><img src="./lecture_11_slides/slide_112182_01-02-23.139.jpg" width="75%" alt="Lecture Video at 01:02:23.139" /></p>

Actually, paradoxically, more recent devices sometimes get worse MFUs. So on the previous generation devices, the H100s, you could sometimes get MFUs above 50%. And the reason for that is because GPUs are getting faster faster than they are getting faster at communicating. When we move from the A100 to the H100, we got roughly a 3x improvement in the theoretical throughput of the compute, but we only got a 2x improvement in the theoretical memory bandwidth.

So there's this growing gap between making GPUs are getting faster really fast, but it's harder to scale the communication between the GPUs. And as a result, we tend to sometimes get worse MFUs actually on more recent generations of devices.


<p align="center"><img src="./lecture_11_slides/slide_113354_01-03-02.245.jpg" width="75%" alt="Lecture Video at 01:03:02.245" /></p>

I intentionally wanted to spend most of the time on those points because those are the ones that you guys are probably going to use in practice. I don't think anyone in this room likely has access to a 10,000 GPU cluster; if you do, come talk to me after class. I would love to be your friend. So those are the ones that you're likely to encounter in practice, like up to many hundreds of GPUs.

But there are these other ones that I just like.


<p align="center"><img src="./lecture_11_slides/slide_114240_01-03-31.808.jpg" width="75%" alt="Lecture Video at 01:03:31.808" /></p>

There are slides here that I think are pretty nice, but it's OK if we don't go through the full details of these; you can check it offline.


<p align="center"><img src="./lecture_11_slides/slide_114374_01-03-36.279.jpg" width="75%" alt="Lecture Video at 01:03:36.279" /></p>

So we said context parallelism is basically splitting on the sequence dimension. We said transformers are operating on sequences, and basically the idea is you've got a long sequence.


<p align="center"><img src="./lecture_11_slides/slide_114658_01-03-45.755.jpg" width="75%" alt="Lecture Video at 01:03:45.755" /></p>

Make different GPUs handle different parts of the sequence. So it's relatively straightforward to ask to chunk up that computation across the sequence dimension.


<p align="center"><img src="./lecture_11_slides/slide_115236_01-04-05.041.jpg" width="75%" alt="Lecture Video at 01:04:05.041" /></p>

<p align="center"><img src="./lecture_11_slides/slide_115250_01-04-05.508.jpg" width="75%" alt="Lecture Video at 01:04:05.508" /></p>

I mean, it does get a little bit hairy inside the MLP because there are weights in there. So you have to have some all-reduce of the gradients like we did in the data parallelism cases.


<p align="center"><img src="./lecture_11_slides/slide_115478_01-04-13.116.jpg" width="75%" alt="Lecture Video at 01:04:13.116" /></p>

<p align="center"><img src="./lecture_11_slides/slide_115574_01-04-16.320.jpg" width="75%" alt="Lecture Video at 01:04:16.320" /></p>

The attention is where things get hairy for sequence parallelism.


<p align="center"><img src="./lecture_11_slides/slide_115750_01-04-22.192.jpg" width="75%" alt="Lecture Video at 01:04:22.192" /></p>

Because if you remember attention, we need to compute these all-pairs interaction between every pair of elements inside the sequence.


<p align="center"><img src="./lecture_11_slides/slide_115894_01-04-26.997.jpg" width="75%" alt="Lecture Video at 01:04:26.997" /></p>

<p align="center"><img src="./lecture_11_slides/slide_116028_01-04-31.468.jpg" width="75%" alt="Lecture Video at 01:04:31.468" /></p>

The QKV projection is easy because that's trivially parallelizable over the sequence, but that core attention matrix actually gets pretty tricky to parallelize.


<p align="center"><img src="./lecture_11_slides/slide_116594_01-04-50.354.jpg" width="75%" alt="Lecture Video at 01:04:50.354" /></p>

There's a lot of details in there; you can check out the paper for more details. The second, which is a little bit conceptually easier, is called Ulysses attention, where you do parallelism over the heads. So remember in a transformer, you're almost always doing multi-head attention where you're computing attention over multiple attention matrices all in parallel.


<p align="center"><img src="./lecture_11_slides/slide_117408_01-05-17.514.jpg" width="75%" alt="Lecture Video at 01:05:17.514" /></p>

As an example, this context parallelism becomes important once you scale up your sequence length to be quite large. So if we go back to this example of Llama3 pretraining, they actually train the model in two stages. The first stage, they go sequence length of 8,000 with no context parallelism whatsoever. And then they have a second stage of training where they crank the sequence length up to 130,000.

At that point, they do 16-way context parallelism. So that means that each of those 131,000 long sequences has 16 GPUs operating on one sequence in parallel. And that's like saying the batch size is $\frac{1}{16}$, because now each batch, each GPU is working on less than one element.


<p align="center"><img src="./lecture_11_slides/slide_118736_01-06-01.824.jpg" width="75%" alt="Lecture Video at 01:06:01.824" /></p>

So that's context parallelism and pipeline parallelism.


<p align="center"><img src="./lecture_11_slides/slide_118854_01-06-05.762.jpg" width="75%" alt="Lecture Video at 01:06:05.762" /></p>

We're going to split across the layers dimension. Intuitively, what you want to do is have a network with a bunch of layers, and we're going to just divide the layers across the GPUs.


<p align="center"><img src="./lecture_11_slides/slide_119108_01-06-14.237.jpg" width="75%" alt="Lecture Video at 01:06:14.237" /></p>

That's actually a very intuitive thing to do. The problem is that there are sequential dependencies because each GPU needs the activations from the previous GPU to continue running the forward pass. During the backward pass, I need the gradients from the upstream layers in order to compute the backward pass.


<p align="center"><img src="./lecture_11_slides/slide_119540_01-06-28.651.jpg" width="75%" alt="Lecture Video at 01:06:28.651" /></p>

So we can draw a diagram like this, where the vertical axis has GPUs 1 to 4, and the horizontal axis is what happens over the course of time. You can see that GPU 1 runs forward, then passes the activations to GPU 2, which passes activations to GPU 3, which passes activations to GPU 4. GPU 4 is lucky; it can do forward and backward all at once, then pass gradients back to GPU 3, back to GPU 2, back to GPU 1.

From this graph, that's obviously really bad because the GPUs are mostly sitting idle. In fact, if you have $N$ GPUs, you're only getting useful work out of them $\frac{1}{N}$ of the time. That means that if we had eight-way pipeline parallelism, your maximum possible MFU at that point is like 12%, which is terrible.


<p align="center"><img src="./lecture_11_slides/slide_120874_01-07-13.163.jpg" width="75%" alt="Lecture Video at 01:07:13.163" /></p>

The trick in pipeline parallelism is to shrink the bubble.


<p align="center"><img src="./lecture_11_slides/slide_121280_01-07-26.709.jpg" width="75%" alt="Lecture Video at 01:07:26.709" /></p>

You want to have less bubble. The way that we do that is running multiple microbatches simultaneously.


<p align="center"><img src="./lecture_11_slides/slide_121746_01-07-42.258.jpg" width="75%" alt="Lecture Video at 01:07:42.258" /></p>

We have four GPUs that are all working in parallel, and then we have four batches of data that are all active at the same time. These batches are color-coded. We see that GPU 1 runs forward on the blue batch, then forward on the yellow batch, then forward on the green batch, then forward on the red batch. While GPU 1 is going forward on the yellow batch, we have passed the activations of the blue batch to GPU 2.

And GPU 2 can now do forward on the blue batch. These things can all cascade down and happen in parallel. In this case, with four-way pipeline parallelism with four microbatches, the max theoretical MFU is just a fraction of this graph, which increases now to 57%, which is pretty good. However, the more microbatches you have, they need to store all the activations in memory.

So now you need to do activation checkpointing. You think like, "Oh crap, how do I tune these things? Should I have more pipeline parallelism? Should I have fewer microbatches?

Should I have more aggressive activation checkpointing?" And then should I also layer data parallelism on top of that? You're going to try to tune all of those knobs to maximize the MFU of your training pipeline.


<p align="center"><img src="./lecture_11_slides/slide_124552_01-09-15.885.jpg" width="75%" alt="Lecture Video at 01:09:15.885" /></p>

<p align="center"><img src="./lecture_11_slides/slide_124626_01-09-18.354.jpg" width="75%" alt="Lecture Video at 01:09:18.354" /></p>

Then the last one is tensor parallelism, and this one you're going to split on the model dimension. Basically, what we're going to do is we have a lot of weight matrices in our model. All those weight matrices are like computing $X W = Y$. That's basically what we're doing over and over again inside of our transformer.


<p align="center"><img src="./lecture_11_slides/slide_125124_01-09-34.970.jpg" width="75%" alt="Lecture Video at 01:09:34.970" /></p>

The idea is that we'll split each weight matrix across GPUs. This is different from FSDP because we're actually splitting a single weight matrix across GPUs, and now there's no communication. We do a block matrix multiply; each GPU is computing a slice of that matrix multiply on the full input. In this case, we split our weight matrix into $W_1$, $W_2$, $W_3$, $W_4$.

Then each GPU just computes a slice of that matrix multiplied to compute a slice of the output.


<p align="center"><img src="./lecture_11_slides/slide_125958_01-10-02.799.jpg" width="75%" alt="Lecture Video at 01:10:02.799" /></p>

The problem is that after you do that forward pass, then you need to gather the activations across all the GPUs to do the next forward pass.


<p align="center"><img src="./lecture_11_slides/slide_126234_01-10-12.008.jpg" width="75%" alt="Lecture Video at 01:10:12.008" /></p>

<p align="center"><img src="./lecture_11_slides/slide_126252_01-10-12.609.jpg" width="75%" alt="Lecture Video at 01:10:12.609" /></p>

<p align="center"><img src="./lecture_11_slides/slide_126268_01-10-13.143.jpg" width="75%" alt="Lecture Video at 01:10:13.143" /></p>

There's a slight trick: if you have two of these layers in sequence, you can actually get away with not gathering in between two layers.


<p align="center"><img src="./lecture_11_slides/slide_126526_01-10-21.751.jpg" width="75%" alt="Lecture Video at 01:10:21.751" /></p>

<p align="center"><img src="./lecture_11_slides/slide_126798_01-10-30.827.jpg" width="75%" alt="Lecture Video at 01:10:30.827" /></p>

If you have two layers, you split the first weight matrix into column-shaped chunks, and then you split the second weight matrix into row-shaped chunks. If you do all this, it all works out magically because of the magic and mystery of block matrix multiplication. The final output you can compute as an inner product like structure of these block matrix multiplies of $Y$ and $U$.


<p align="center"><img src="./lecture_11_slides/slide_127220_01-10-44.908.jpg" width="75%" alt="Lecture Video at 01:10:44.908" /></p>

So you basically can have two layers of matrix multiply that are split across multiple GPUs, and then they only need to communicate. at the end of every two layers. And this actually works out nicely because remember transformers have a two layer MLP in the FFN. So this is a really nice trick that plays really nicely into the two-layer MLPs that transformers always have.

It's pretty common in big transformers to use tensor parallelism, like two-layer tensor parallelism trick on the MLP in a transformer.


<p align="center"><img src="./lecture_11_slides/slide_128110_01-11-14.604.jpg" width="75%" alt="Lecture Video at 01:11:14.604" /></p>

<p align="center"><img src="./lecture_11_slides/slide_128128_01-11-15.204.jpg" width="75%" alt="Lecture Video at 01:11:15.204" /></p>

<p align="center"><img src="./lecture_11_slides/slide_128158_01-11-16.205.jpg" width="75%" alt="Lecture Video at 01:11:16.205" /></p>

<p align="center"><img src="./lecture_11_slides/slide_128170_01-11-16.606.jpg" width="75%" alt="Lecture Video at 01:11:16.606" /></p>

That's basically all of our mechanisms for splitting up computation across GPUs.


<p align="center"><img src="./lecture_11_slides/slide_128292_01-11-20.676.jpg" width="75%" alt="Lecture Video at 01:11:20.676" /></p>

<p align="center"><img src="./lecture_11_slides/slide_128412_01-11-24.680.jpg" width="75%" alt="Lecture Video at 01:11:24.680" /></p>

Which one is the best? The actual answer is all of them. In practice, we're going to use ND parallelism. We saw already an example of two-dimensional parallelism with HSRF.

In practice, the current state of the art is like four-dimensional parallelism. If we go back to Llama, we see that they are training on their biggest training run with 16,000 GPUs. If you're careful, different mechanisms of parallelism have different communication requirements.


<p align="center"><img src="./lecture_11_slides/slide_129678_01-12-06.923.jpg" width="75%" alt="Lecture Video at 01:12:06.923" /></p>

That's basically a whirlwind tour of large-scale distributed training. The takeaway for today is that an individual GPU is basically a generalizable parallel computing machine. A GPU cluster is a giant massively parallel machine with tens of thousands, maybe hundreds of thousands of individual GPUs, and we want to program it as one big unit. So the next time you're going out and training on tens of thousands of GPUs, hope you keep this in mind.


<p align="center"><img src="./lecture_11_slides/slide_130908_01-12-47.964.jpg" width="75%" alt="Lecture Video at 01:12:47.964" /></p>

And let me know so I can borrow your tens of thousands of GPUs.
